# Limpieza de datos - NHANES (2005-2014)
Cuaderno de evidencia de la Fase 1.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent / 'src'))
from repositorios.cargador import CargadorDataset
from repositorios.nhanes import NhanesSchemaMapper, NhanesFusion
from utilidades.constantes import CARPETA_RAW, ARCHIVOS_NHANES, COLUMNAS_CURADAS
listaDfs = []
for n in ARCHIVOS_NHANES:
    df = CargadorDataset(CARPETA_RAW / n).cargar()
    listaDfs.append(df)
    print(n, df.shape)


In [ ]:
fusion = NhanesFusion(NhanesSchemaMapper(COLUMNAS_CURADAS))
fusionado, comunes = fusion.fusionar(listaDfs)
print('Filas:', len(fusionado), 'Columnas:', len(comunes))
print('Duplicados SEQN:', int(fusionado['SEQN'].duplicated().sum()))


In [ ]:
from servicios.limpieza import TratadorDuplicados, NormalizadorCodigosNhanes, TypeCleaner, MissingValueHandler
from servicios.calidad import Diagnosticador, AnalizadorOutliers, Validador
from utilidades.constantes import COLUMNAS_CATEGORICAS, COLUMNAS_NUMERICAS
df = TratadorDuplicados('SEQN').quitar(fusionado.copy())
print('Nulos antes:', int(df.isna().sum().sum()))
df = NormalizadorCodigosNhanes().normalizar(df)
df = TypeCleaner(COLUMNAS_CATEGORICAS, COLUMNAS_NUMERICAS).limpiar(df)
df = MissingValueHandler(COLUMNAS_NUMERICAS, COLUMNAS_CATEGORICAS).imputar(df)
print('Nulos despues:', int(df.isna().sum().sum()))


In [ ]:
analizador = AnalizadorOutliers(COLUMNAS_NUMERICAS, k=1.5)
limites = analizador.revisarIQR(df)
df = analizador.marcar(df, limites)
Validador().validar(df, COLUMNAS_NUMERICAS)


## Conclusion
- Integracion por interseccion de columnas: 26 columnas comunes, 0 duplicados de SEQN.
- Limpieza: nulos imputados con mediana/moda, tipos convertidos, outliers marcados con IQR (flag).
- Salida: outputs/cleaned/nhanes_master_cleaned.csv.gz (50965 x 27).
